# 01 · Hello World, twice

## Goal

Create the smallest possible agent two ways — once as YAML via `pac copilot`,
once by hand in the portal — publish each, invoke it over the API, and
delete it. You leave this notebook knowing exactly what a `pac copilot
init` scaffold contains, and having made (and understood) the one
irreversible choice every later notebook assumes: the GitHub Copilot
harness.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("TENANT_ID", "DATAVERSE_ENV_ID", "APP_CLIENT_ID", "DELEGATED_CLIENT_ID")
print("settings loaded")


## Concept

**Finding #1, stated plainly:** the harness — GitHub Copilot (GHCP) or the
standard Copilot Studio harness — is chosen at agent creation and **cannot
be changed**. There is no "start standard, upgrade later" path; moving
harnesses means rebuilding the agent. This curriculum uses GHCP throughout
(GA 3 Aug 2026) because it's where the Python sandbox, native Office file
handling, and markdown skills (`07`-`08`) live. If your org has a reason to
prefer the standard harness, know that before you run the cell below —
switching later costs you everything in `agents/contract-renewal-desk`.

**Why build it twice:** the portal and the YAML describe the same object.
Seeing both once means every later notebook's YAML diff reads as "this is
what I'd have clicked," not as opaque config.


## Build


### 1a — the YAML path


In [ ]:
from pathlib import Path
from csx.pac import copilot_init, copilot_pack

workspace = Path("../agents/hello-world-throwaway")
copilot_init(workspace, authoring_mode="cli-copilot")

# Inspect what got scaffolded before changing anything
for p in sorted(workspace.rglob("*")):
    print(p.relative_to(workspace))


In [ ]:
# Set the harness explicitly — this is the irreversible line.
copilot_yaml = workspace / "copilot.yaml"
text = copilot_yaml.read_text()
text = text.replace("harness: standard", "harness: github-copilot") if "harness:" in text else text + "\nharness: github-copilot\n"
copilot_yaml.write_text(text)
print(copilot_yaml.read_text())


In [ ]:
from csx.pac import copilot_push, solution_import

copilot_push(workspace)  # requires the delegated or application auth profile from 00


### 1b — the portal path

In the Copilot Studio portal: **Create > New agent > Describe > Build in GitHub Copilot harness**. Give it a throwaway name. This is the one deliberate click-through in the whole curriculum — the point is to *feel* the mapping to the YAML above, not to avoid the portal forever.


In [ ]:
from csx.checkpoint import checkpoint
import subprocess, json

def probe_portal_agent():
    result = subprocess.run(["pac", "copilot", "list", "--json"], capture_output=True, text=True)
    agents = json.loads(result.stdout) if result.returncode == 0 else []
    return next((a for a in agents if "hello" in a.get("name", "").lower() and a.get("schemaName") != "crd_hello-world-throwaway"), None)

portal_agent = checkpoint(
    name="Portal-built hello-world agent exists",
    probe=probe_portal_agent,
    remediation="In the Copilot Studio portal: Create > New agent > Build in GitHub Copilot harness. Publish it.",
)


## Verify

Same harness, same golden set, every notebook.


Invoke both agents over the API — delegated locally, and once more with the application/SP path, so the CI-safe route is proven before anything depends on it.


In [ ]:
from csx.clients import get_copilot_client

client_delegated = get_copilot_client(settings, delegated=True)
reply = client_delegated.ask_question("Hello, what can you do?")
print("delegated:", reply.text[:200])

client_application = get_copilot_client(settings, delegated=False)
reply2 = client_application.ask_question("Hello, what can you do?")
print("application (CI path):", reply2.text[:200])


## Cost


In [ ]:
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
# Building + publishing + two invocations of two throwaway agents. Record
# whatever the admin center analytics showed you for this session.
meter.report_cost("01", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=12, note="2x hello-world build+publish+2 invokes")


## Teardown


In [ ]:
import subprocess
# Delete both throwaway agents — the real spine agent is created fresh in 02,
# not reused from here, so nothing about these two survives.
subprocess.run(["pac", "copilot", "delete", "--name", "crd_hello-world-throwaway"], check=False)
subprocess.run(["pac", "copilot", "delete", "--name", "crd_hello-world-portal"], check=False)
print("both hello-world agents deleted")
